In [ ]:
import otter
grader = otter.Notebook("hw09.ipynb")

<div class="alert alert-success">

#### Homework 9 Supplemental Notebook
    
# Gradients and Gradient Descent

### EECS 245, Fall 2025 at the University of Michigan
    
</div>

### Instructions

Most homeworks will have Jupyter Notebooks, like this one, designed to supplement the theoretical problems. 

To write and run code in this notebook, you have two options:

1. **Use the EECS 245 DataHub.** To do this, click the link provided in the Homework 9 PDF. Before doing so, read the instructions on the [**Tech Support**](https://eecs245.org/tech-support/#option-1-using-the-eecs-245-datahub) page on how to use the DataHub.
1. **Set up a Jupyter Notebook environment locally, and use `git` to clone our course repository.** For instructions on how to do this, see the [**Tech Support**](https://eecs245.org/tech-support) page of the course website.

Problem 5 is contained **entirely** in this notebook, and is **entirely autograded**. To receive credit for Problem 5, submit your completed notebook to the Homework 9, Problem 5 Code autograder on Gradescope. Your submission time for Homework 9 is the **latter** of your PDF and code submission times.

This particular programming problem has **no hidden test cases** – the test cases on Gradescope are exactly the ones available in your notebook.

In [ ]:
# Run this cell.
import numpy as np
import pandas as pd
import time

pd.options.plotting.backend = "plotly"

import plotly.express as px
import plotly.io as pio
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

# Set default layout for all plotly figures
import plotly.graph_objs as go

custom_template = go.layout.Template(pio.templates["plotly_white"])
custom_template.layout.plot_bgcolor = "white"
custom_template.layout.paper_bgcolor = "white"
custom_template.layout.margin = dict(l=60, r=60, t=60, b=60)
custom_template.layout.width = 700
custom_template.layout.font = dict(
    family="Palatino Linotype, Palatino, serif",
    color="black"
)

pio.templates["custom"] = custom_template
pio.templates.default = "custom"

import hw09_util as util

## Problem 5: Implementing Gradient Descent (12 pts; all autograded)

---

In this problem, you'll implement gradient descent yourself, to find optimal model parameters.

To set up the problem, let's load in the now-familiar commute times dataset from lecture.

In [ ]:
df = pd.read_csv('data/commute-times.csv')
df.head()

One of our running examples has been to build a simple linear regression model of the form $h(x_i) = w_0 + w_1 x_i$, that predicts commute time in `minutes` given `departure_hour`.

In [ ]:
line_fig = px.scatter(df, x='departure_hour', y='minutes').update_layout(xaxis_title='Home Departure Time (AM)', yaxis_title='Minutes', title='Commuting Time vs. Home Departure Time')
line_fig.show()

The "default" approach has been to choose squared loss, meaning that we choose the intercept, $w_0^*$, and slope, $w_1^*$, that minimize **mean squared error**:

$$R_\text{sq}(w_0, w_1) = \frac{1}{n} \sum_{i = 1}^n \left( y_i - (w_0 + w_1 x_i) \right)^2$$

We solved for $w_0^*$ and $w_1^*$ by hand using calculus in [Chapter 1.4](https://notes.eecs245.org/supervised-learning/simple-linear-regression/), and in [Chapter 2.10](https://notes.eecs245.org/vectors-and-matrices/projection-2/) and [Chapter 3.1](https://notes.eecs245.org/multiple-linear-regression/regression-using-linear-algebra/) we found an equivalent solution using linear algebra (which is how we found the normal equations). For reference, we'll calculate the slope and intercept that minimize mean squared error below.

In [ ]:
from sklearn.linear_model import LinearRegression
baseline = LinearRegression()
baseline.fit(df[['departure_hour']], df['minutes'])
w_squared_loss = baseline.intercept_, baseline.coef_[0]
w_squared_loss

So, by minimizing mean squared error, we have $w_0^* = 142.45$ and $w_1^* = -8.19$. Keep these values in mind throughout the question. For reference, here's what <b><span style="color:orange">the line that minimizes mean squared error</span></b> looks like:

In [ ]:
line_fig.add_trace(
    go.Scatter(x=[5.5, 11.5], y=[baseline.predict([[5.5]])[0], baseline.predict([[11.5]])[0]], mode='lines', line=dict(color='orange'), name='Best Line (MSE)')
)

line_fig.show()

<br>

Another approach is to choose absolute loss, meaning that we choose intercept and slope that minimize **mean absolute error**:

$$R_\text{abs}(w_0, w_1) = \frac{1}{n} \sum_{i = 1}^n \left| y_i - (w_0 + w_1 x_i) \right|$$

We can't directly use gradient descent to minimize $R_\text{abs}$ since it's not differentiable. In Homework 2, Problem 5, you implemented a brute-force computational routine that found the optimal slope and intercept in $O(n^3)$ time. 

<br>

What we'll explore here is a **new** loss function, Tukey's loss function, named after [John Tukey](https://en.wikipedia.org/wiki/John_Tukey), the creator of the box plot. It is defined as follows:

$$L_T(y_i, h(x_i)) = \begin{cases} 1 - \left( 1 - \left( \frac{y_i - h(x_i)}{50} \right)^2 \right)^3 && \text{if} \: |y_i - H(x_i)| \leq 50, \\ 1 && \text{otherwise} \end{cases}$$

To make sense of how the loss function behaves, let's graph it.

In [ ]:
def tukey(y_actual, y_pred, c=50):
    error = y_actual - y_pred
    if np.abs(error) <= c:
        return 1 - (1 - (error / c) ** 2) ** 3
    else:
        return 1

hs = np.linspace(-200, 200, 10000)
fig = px.line(x=hs, y=[tukey(0, h) for h in hs], title='Tukey Loss').update_layout(xaxis_title='h', yaxis_title='L(0, h)')
fig.show()

Note that Tukey loss is defined **piecewise**. 
- For predictions in which $|y_i - h(x_i)|$ is more than 50, the loss is capped at 1, resulting in a loss function that is _very_ robust to outliers (unlike squared loss, which is influenced heavily by outliers). 
- For predictions in which $|y_i - h(x_i)| \leq 50$, the loss looks very similar to squared loss. 

The choice of 50 as the threshold was arbitrary; we could have chosen a different transition point (and, once you finish the question, it's worth investigating how your results would have changed if 50 was replaced with 5 or 100).

You'll also notice that the "handoff" from the curved part to the flat part at $h = \pm 50$ is smooth, meaning Tukey loss is differentiable, unlike absolute loss. This will be important!

Let's continue to consider the simple linear regression model, $h(x_i) = w_0 + w_1 x_i$, where $x_i$ is a scalar, not a vector. In that case, the empirical risk $R_T$ using Tukey loss looks like:

$$R_T(w_0, w_1) = \frac{1}{n} \sum_{i = 1}^n  \begin{cases} 1 - \left( 1 - \left( \frac{y_i - (w_0 + w_1 x_i)}{50} \right)^2 \right)^3 && \text{if} \: |y_i - (w_0 + w_1 x_i)| \leq 50, \\ 1 && \text{otherwise} \end{cases}$$

Let's graph the loss surface, using some help from the helper functions we've defined in `hw09_util.py`. Note that we're specifying that we want the $x$'s and $y$'s to come from the `departure_hour` and `minutes` columns in `df`, respectively.

In [ ]:
fig = util.draw_loss_surface_tukey_commute(xs=df['departure_hour'], ys=df['minutes'])
fig.show()

Let's use gradient descent to find the intercept, $w_0^*$, and slope, $w_1^*$, that minimize the loss surface above. That is, let's find the best intercept and slope to use in a simple linear model that predicts `minutes` using `departure_hour`, using Tukey loss. We can think of this problem as trying to find the vector, $\vec w^*$, that minimizes $R_T(\vec w)$, where $\vec w = \begin{bmatrix} w_0 \\ w_1 \end{bmatrix}$.

The gradient descent update rule is as follows:

$$\vec w^{(t+1)} = \vec w^{(t)} - \alpha \nabla R_T( \vec w^{(t)})$$

where $\vec w^{(t)}$ is our guess for the minimizing $\vec w^*$ at timestep $t$. To start the process, we'll need to decide on an initial guess, $\vec w^{(0)}$, and a step size, $\alpha$, which we will do later.

But, more crucially, to run gradient descent ourselves, we'll need to be able to compute $\nabla R_T(\vec w^{(t)})$, i.e. the **gradient** of $R_T$ at any point $\vec w$.

### Problem 5a) (7 pts)

As a refresher, the empirical risk function that we're trying to minimize is:

$$R_T(\vec w) = R_T(w_0, w_1) = \frac{1}{n} \sum_{i = 1}^n  \begin{cases} 1 - \left( 1 - \left( \frac{y_i - (w_0 + w_1 x_i)}{50} \right)^2 \right)^3 && \text{if} \: |y_i - (w_0 + w_1 x_i)| \leq 50, \\ 1 && \text{otherwise} \end{cases}$$

Complete the implementation of the function `tukey_gradient`, which takes in:
- `w`, an **array of length 2**, corresponding to a value of $w_0$ and a value of $w_1$, and
- `xs` and `ys`, two Series/1D arrays with the same length, corresponding to sequences of $x$-values (like `df['departure_hour']`) and $y$-values (like `df['minutes']`), respectively.

`tukey_gradient` should return an **array of length 2**, containing the value of the gradient of $R_T$, evaluated at the point `w` that is passed in. Example behavior is given below.

```python
# This says that dR/dw0, when (w0, w1) = (100, -5), is -0.02389409, and
# dR/dw1, when (w0, w1) = (100, -5), is -0.20369193.
>>> tukey_gradient(np.array([100, -5]), xs=df['departure_hour'], ys=df['minutes'])
array([-0.02389409, -0.20369193])
```

Some guidance:
- Remember, the gradient of a function is a vector of partial derivatives. So, $\nabla R_T(\vec w) = \begin{bmatrix} \frac{\partial R}{\partial w_0} \\ \frac{\partial R}{\partial w_1} \end{bmatrix}$.
- Because $R_T$ is a piecewise function, both partial derivatives will also be piecewise, using the same condition as in $R_T$. So, your definition of `tukey_gradient` can involve `if`-statements.
- You'll need to do a substantial amount of math on-paper to complete this implementation, and accurately translate it to code. `for`-loops are fine in your implementation.
    - Back in Chapter 1.4 (which you should review for this!), we stressed that the derivative of a sum of functions is equal to the sum of the derivatives of those functions. In other words, $\frac{\partial R_T}{\partial w_0} = \frac{1}{n} \sum_{i = 1}^N \frac{\partial L_T}{\partial w_0}$. Given this, it's easiest to start with finding the partial derivatives of just the loss function $L_T$ with respect to $w_0$ and $w_1$, and summing these partial derivatives in a `for`-loop.
    - When computing the partial derivatives of $L_T$ with respect to the two parameters, you may find it helpful to perform a substitution, e.g. $e_i = y_i - (w_0 + w_1 x_i)$, and then use the chain rule for scalar-to-scalar functions, i.e. $\frac{\partial L_T}{\partial w_0} = \frac{\partial L_T}{\partial e_i} \cdot \frac{\partial e_i}{\partial w_0}$.
    - $\frac{\partial R}{\partial w_0}$ and $\frac{\partial R}{\partial w_1}$ will look very similar, but with one small difference.

In [ ]:
def tukey_gradient(w, xs, ys):
    ...

# Feel free to change these inputs to make sure your functions work correctly.
tukey_gradient(np.array([100, -5]), xs=df['departure_hour'], ys=df['minutes'])

In [ ]:
grader.check("p05a")

### Problem 5b) (5 pts)

Complete the implementation of the function `run_gradient_descent_tukey`, which takes in:
- `w_initial`, an **array of length 2**, corresponding to an initial guess, $\vec w^{(0)}$, of the minimizer.
- `alpha`, a positive number corresponding to a step size/learning rate.
- `xs` and `ys`, two Series/1D arrays with the same length, corresponding to sequences of $x$-values (like `df['departure_hour']`) and $y$-values (like `df['minutes']`), respectively.
- `tol`, a float representing the **convergence criteria**, the default value of which will be set to `0.0001` (i.e. $10^{-3}$).
- `verbose`, a Boolean flag.

`run_gradient_descent_tukey` should run as many iterations of gradient descent as necessary, terminating once $\lVert \nabla R_T(\vec w^{(t)}) \rVert_2 \leq \text{tol}$, i.e. once the $L_2$ norm of `tukey_gradient(w, xs, ys)` (where `w` is the current guess of $\vec w^*$) is less than `tol`.
`run_gradient_descent_tukey` should return a 2D array containing all vectors $\vec w^{(t)}$ that were visited by gradient descent. In other words, it should return a 2D array of shape `(num_iterations, 2)`, where row 0 is $\vec w^{(0)}$ (the initial guess), row 1 is $\vec w^{(1)}$, row 2 is $\vec w^{(2)}$, and so on, until row -1 is our final guess of $\vec w^*$.

If `verbose=True`, then on iterations 0, 1000, 2000, and so on, use the `display` function to show the iteration number $t$, the value of $\vec w^{(t)}$, the value of  $R_T(\vec w^{(t)})$, and the value of $\lVert \nabla R_T(\vec w^{(t)}) \rVert_2$ (i.e. the norm of the gradient vector) at the current iteration. You can compute $R_T(\vec w^{(t)})$ using the `tukey` function defined at the top of the notebook, or using `util.empirical_risk`. We won't test your code with `verbose=True`, so you have some flexibility in how to implement it, but you'll need to complete this step in order for the remainder of the problem to make sense.

Finally, if more than 50,000 iterations have been completed (including the initial guess), terminate the algorithm and return an array of shape `(50000, 2)` containing the 50,000 vectors $\vec w^{(0)}, \vec w^{(1)}, ... \vec w^{(49,999)}$ that were visited by the algorithm.

Example behavior is given below.

```python
>>> path = run_gradient_descent_tukey(w_initial=np.array([100, 0]), 
                                      alpha=10, xs=df['departure_hour'], ys=df['minutes'])

# Our guess of w^*, the optimal parameter vector.
>>> path[-1]
array([121.5414978 ,  -5.85456359])

# The number of steps it took to reach the above guess, including the first and last guesses.
>>> len(path)
7616
```

In [ ]:
def run_gradient_descent_tukey(w_initial, alpha, xs, ys, tol=0.0001, verbose=False):
    ...

# Feel free to change these inputs to make sure your functions work correctly.
path = run_gradient_descent_tukey(w_initial=np.array([100, 0]), 
                           alpha=10, 
                           xs=df['departure_hour'], 
                           ys=df['minutes'], 
                           verbose=True)
path[-1]

In [ ]:
grader.check("p05b")

Nice work! Let's experiment with what you've done. The expression below will call your implementation of `run_gradient_descent_tukey`, and draw out the path that gradient descent took to minimize $R_T(\vec w)$ on the loss surface of $R_T(\vec w)$ itself, using an initial guess of $\vec w^{(0)} = \begin{bmatrix} 100 \\ 0 \end{bmatrix}$ and $\alpha = 10$ (as in the example output provided at the start of Problem 5b). If you hover over a point in gold, you'll see its iteration number, $t$. (Scroll to the right to see the contour plot.)

In [ ]:
path = run_gradient_descent_tukey(w_initial=np.array([100, 0]), alpha=10, xs=df['departure_hour'], ys=df['minutes'])
fig = util.draw_loss_surface_tukey_commute(xs=df['departure_hour'], ys=df['minutes'], path=path)
fig.show()

You should notice that within a few steps, gradient descent gets down to the valley, but it takes thousands more iterations to inch sufficiently close to the true minimum. If you called `run_gradient_descent_tukey` with `verbose=True` above, you should have seen that the value of $R_T(\vec w^{(t)})$ decreased very slowly every 1000 iterations, and the norm of the gradient vector very, very slowly approached 0. If we settled for a greater tolerance – say, $0.0005$ instead of $0.0001$, we'd have converged quicker:

In [ ]:
run_gradient_descent_tukey(w_initial=np.array([100, 0]), 
                           alpha=10, 
                           xs=df['departure_hour'], 
                           ys=df['minutes'], 
                           tol=0.0005,
                           verbose=True)

But, the resulting $\vec w^*$ of $\begin{bmatrix} 105.234 \\ -3.978 \end{bmatrix}$ is quite far from what we got when using a tolerance of $0.0001$, which gave us $\begin{bmatrix} 121.541 \\ -5.855 \end{bmatrix}$. Why do you think this is happening?

Instead of weakening our tolerance, perhaps we can try a different learning rate:

In [ ]:
run_gradient_descent_tukey(w_initial=np.array([100, 0]), 
                           alpha=15, 
                           xs=df['departure_hour'], 
                           ys=df['minutes'], 
                           tol=0.0001,
                           verbose=True)

If you've implemented everything correctly, you should see that the above call to gradient descent maxes out at 50,000 iterations. Something strange must be going on, since the guesses of $\vec w^{(t)}$ seem to be relatively constant every 1000 iterations, as do the values of $R_T(\vec w^{(t)})$ and $\lVert \nabla R_T(\vec w^{(t)}) \rVert_2$. What's going on? Make an educated guess, then run the cell below. (It should take ~30 seconds to render.)

In [ ]:
path = run_gradient_descent_tukey(w_initial=np.array([100, 0]), alpha=15, xs=df['departure_hour'], ys=df['minutes'])
fig = util.draw_loss_surface_tukey_commute(xs=df['departure_hour'], ys=df['minutes'], path=path)
fig.show()

It seems that a learning rate of $\alpha = 15$ is too large, and results in oscillatory behavior between iterations $t$ and $t+1$. Since we're only printing every 1000 iterations when `verbose=True`, we only get to see one of the two oscillatory states (e.g., in the sequence $a$, $b$, $a$, $b$, $a$, $b$, ..., if we sample just the even positions, we'd think the entire sequence was made up of $b$'s).

Maybe an even larger learning rate will work better – let's try $\alpha = 50$.

In [ ]:
run_gradient_descent_tukey(w_initial=np.array([100, 0]), 
                           alpha=50, 
                           xs=df['departure_hour'], 
                           ys=df['minutes'], 
                           tol=0.0001,
                           verbose=True)

That was quick... what happened?

In [ ]:
path = run_gradient_descent_tukey(w_initial=np.array([100, 0]), alpha=50, xs=df['departure_hour'], ys=df['minutes'])
fig = util.draw_loss_surface_tukey_commute(xs=df['departure_hour'], ys=df['minutes'], path=path)
fig.show()

Why did gradient descent get stuck up top? Something you may have noticed is that $R_T(\vec w)$ is **not convex**. This means that there can be regions in which the gradient of $R_T(\vec w)$ is 0 that don't correspond to a global minimum (which isn't possible for convex functions). The step size of $\alpha = 50$ is so large that, after a few iterations, gradient descent lands us in the flat region, and once a $\vec w^{(t)}$ lands there, $\nabla R_T(\vec w^{(t)}) = \begin{bmatrix} 0 \\ 0 \end{bmatrix}$, terminating the algorithm instantly.

So, to summarize:
- Due to the nature of the loss surface, gradient descent can get _near_ the minimum fairly quickly, but may take many iterations to actually converge.
- If we choose the step size to be too large, gradient descent may oscillate infinitely, "bouncing" over the minimum.
- Since the loss surface is not convex, gradient descent can get "trapped" in flat regions and terminate mistakenly.

Hopefully, you now have a better understanding of how gradient descent works and how it can go wrong.

Finally, let's actually take a look 👀 at the line that minimizes mean Tukey loss on the commute times dataset.

In [ ]:
w0, w1 = run_gradient_descent_tukey(w_initial=np.array([100, 0]), 
                                    alpha=10, xs=df['departure_hour'], ys=df['minutes'],
                                    verbose=True)[-1]
w0, w1

In [ ]:
line_fig.add_trace(
    go.Scatter(x=[5.5, 11.5], y=[w0 + w1 * 5.5, w0 + w1 * 11.5], mode='lines', line=dict(color='purple'), name='Best Line (Mean Tukey Loss)')
)

line_fig.show()

Knowing what you know about Tukey loss, how does the <b><span style="color:orange">line that minimizes mean squared error</span></b> compare to the <b><span style="color: purple">line that minimizes mean Tukey loss</span></b>? You don't need to write the answer to this question – or any of the other thought-provoking questions in this notebook – anywhere, but you _should_ think about them.

## Finish Line 🏁

To get credit for Problem 5, submit this notebook to Gradescope.
1. Select `Kernel -> Restart & Run All` to ensure that you have executed all cells, including the test cells.
2. Read through the notebook to make sure everything is fine and all public tests passed.
3. Run the cell below to run all tests, and make sure that they all pass.
4. Download your notebook using `File -> Download`, then upload your notebook to Gradescope under "Homework 9, Problem 5 Code".
5. Stick around for a few minutes while the Gradescope autograder grades your work. Make sure you see that all **public tests** have passed on Gradescope. **This particular problem has no hidden test cases, so you should see a full score shortly after submitting.**